# Week 1, Learn part 1: pandas refresh

Six pandas moves cover most of the data work in this runbook. Each section explains the idea, maps it to the BI equivalent you already know, and runs it on the NC PLACES data you downloaded.

| pandas move | BI equivalent |
| --- | --- |
| `read_csv` with `dtype` | Setting column types in Power Query |
| `query` / `loc` | Filters and slicers |
| `pivot_table` / `melt` | Pivot / unpivot |
| `groupby().agg()` | Measures grouped by a dimension |
| `merge(validate=)` | Relationships with cardinality |
| `describe()`, `value_counts()` | Profiling view |

**Before running:** the files `data/raw/places_tract_nc.csv` and `data/raw/places_county_nc.csv` must exist, and this notebook must sit in the `notebooks/` folder.

**Goal:** by the end, rebuild the wide tract table from the raw file without looking back, in under 10 minutes.

## Setup

In [11]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW = Path("../data/raw")
TRACT_FILE = RAW / "places_tract_nc.csv"
COUNTY_FILE = RAW / "places_county_nc.csv"

for f in (TRACT_FILE, COUNTY_FILE):
    if not f.exists():
        raise FileNotFoundError(f"Missing {f.resolve()}: download it from the CDC PLACES portal first.")
print("Files found. pandas", pd.__version__)

Files found. pandas 3.0.6


Files exported from the PLACES portal use mixed-case column names (`MeasureId`). Files pulled from the API use lowercase (`measureid`). This helper makes both look the same, so the rest of the notebook works either way.

In [21]:
PORTAL_NAMES = ["Year", "StateAbbr", "StateDesc", "CountyName", "CountyFIPS", "LocationName",
                "DataSource", "Category", "Measure", "Data_Value_Unit", "Data_Value_Type",
                "Data_Value", "Low_Confidence_Limit", "High_Confidence_Limit",
                "TotalPopulation", "TotalPop18plus", "LocationID", "CategoryID",
                "MeasureId", "DataValueTypeID", "Short_Question_Text", "Geolocation"]
LOOKUP = {name.lower(): name for name in PORTAL_NAMES}

def standardise_columns(df):
    return df.rename(columns=lambda c: LOOKUP.get(c.lower(), c))

## 1. Read with explicit types

A census tract ID like `37183052401` is an **identifier, not a number**. NC's state code (37) has no leading zero, so reading it as an integer looks harmless here. But states like Alabama (`01`) lose the zero, and every join on that ID then silently fails. Make string IDs a habit now.

The cell below shows the problem on a tiny example first.

In [13]:
demo = pd.DataFrame({"tract": ["01001020100", "37183052401"]})
demo.to_csv("_demo.csv", index=False)

print(pd.read_csv("_demo.csv"))                           # zero lost: 1001020100
print(pd.read_csv("_demo.csv", dtype={"tract": str}))     # zero kept
Path("_demo.csv").unlink()

         tract
0   1001020100
1  37183052401
         tract
0  01001020100
1  37183052401


In [14]:
ID_COLS = {c: str for c in ["LocationName", "LocationID", "CountyFIPS"]}
ID_COLS.update({c.lower(): str for c in list(ID_COLS)})   # cover API lowercase names too

df = standardise_columns(pd.read_csv(TRACT_FILE, dtype=ID_COLS))
print(df.shape)
df.dtypes

(105878, 24)


Year                            int64
StateAbbr                         str
StateDesc                         str
CountyName                        str
CountyFIPS                        str
LocationName                      str
DataSource                        str
Category                          str
Measure                           str
Data_Value_Unit                   str
Data_Value_Type                   str
Data_Value                    float64
Data_Value_Footnote_Symbol    float64
Data_Value_Footnote           float64
Low_Confidence_Limit          float64
High_Confidence_Limit         float64
TotalPopulation                   str
TotalPop18plus                    str
Geolocation                       str
LocationID                        str
CategoryID                        str
MeasureId                         str
DataValueTypeID                   str
Short_Question_Text               str
dtype: object

## 2. Filter with `query` and `loc`

`query` reads like SQL and is easy to review. `loc` is better when a condition is built in code or you also want to pick columns. Both return the same rows.

In [15]:
keep = ["DIABETES", "OBESITY", "BPHIGH"]
cols = ["LocationName", "CountyName", "MeasureId", "Data_Value",
        "Low_Confidence_Limit", "High_Confidence_Limit", "TotalPopulation"]

via_query = df.query("MeasureId in @keep")[cols]         # @ refers to a Python variable
via_loc = df.loc[df["MeasureId"].isin(keep), cols]

assert via_query.equals(via_loc)
sub = via_loc.copy()
print(sub.shape)
sub["MeasureId"].value_counts()

(7941, 7)


MeasureId
OBESITY     2647
DIABETES    2647
BPHIGH      2647
Name: count, dtype: int64

Combining conditions: use `&` (and), `|` (or), `~` (not), with each condition in brackets.

In [16]:
wake_diabetes = sub.loc[(sub["CountyName"] == "Wake") & (sub["MeasureId"] == "DIABETES")]
print(len(wake_diabetes), "Wake County tracts")
wake_diabetes.sort_values("Data_Value", ascending=False).head()

228 Wake County tracts


,LocationName,CountyName,MeasureId,Data_Value,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation
96093,37183052001,Wake,DIABETES,18.9,16.8,21.2,"3,782"
62843,37183052002,Wake,DIABETES,17.0,14.7,19.3,"5,085"
83789,37183054306,Wake,DIABETES,16.6,14.7,18.8,"3,635"
66310,37183052101,Wake,DIABETES,16.3,14.3,18.5,"8,750"
105527,37183054303,Wake,DIABETES,16.1,14.1,18.4,"2,783"


## 3. Long to wide with `pivot_table`, and back with `melt`

PLACES has **one row per tract per measure** (long). To compare measures side by side you want **one row per tract, one column per measure** (wide), the same as unpivoted vs pivoted data in Power Query.

Watch out: `pivot_table` silently **averages** duplicate rows. Check that each tract and measure pair is unique first.

In [17]:
dupes = sub.duplicated(["LocationName", "MeasureId"])
assert not dupes.any(), f"{dupes.sum()} duplicate tract/measure rows: check for multiple years or value types"

wide = sub.pivot_table(
    index=["LocationName", "CountyName", "TotalPopulation"],
    columns="MeasureId",
    values="Data_Value",
).reset_index()
wide.columns.name = None
print(wide.shape)
wide.head()

(2647, 6)


,LocationName,CountyName,TotalPopulation,BPHIGH,DIABETES,OBESITY
0,37001020100,Alamance,"4,040",39.6,13.7,40.7
1,37001020200,Alamance,"4,137",44.7,20.1,49.3
2,37001020301,Alamance,"4,213",45.3,19.4,47.4
3,37001020302,Alamance,"3,974",40.1,17.4,46.4
4,37001020400,Alamance,"6,556",44.2,18.4,48.0


`melt` goes the other way. You'll use it when plotting all three measures in one seaborn call.

In [18]:
long = wide.melt(id_vars=["LocationName", "CountyName"], value_vars=keep,
                 var_name="measure", value_name="pct")
print(long.shape)                       # should be rows in wide x 3
long.head()

(7941, 4)


,LocationName,CountyName,measure,pct
0,37001020100,Alamance,DIABETES,13.7
1,37001020200,Alamance,DIABETES,20.1
2,37001020301,Alamance,DIABETES,19.4
3,37001020302,Alamance,DIABETES,17.4
4,37001020400,Alamance,DIABETES,18.4


## 4. Group and aggregate

Named aggregation (`new_name=(column, function)`) keeps outputs readable, like naming a measure in DAX.

In [19]:
by_county = wide.groupby("CountyName").agg(
    tracts=("LocationName", "count"),
    population=("TotalPopulation", "sum"),
    diabetes_median=("DIABETES", "median"),
    diabetes_max=("DIABETES", "max"),
)
by_county.sort_values("diabetes_median", ascending=False).head(10)

,tracts,population,diabetes_median,diabetes_max
CountyName,,,,
Warren,7,"2,2151,3162,4263,4692,2273,7263,263",21.50,23.1
Northampton,10,"1,4421,4321,6471,0432,1151,8251,6441,1933,5921...",21.15,24.9
Halifax,16,"2,9085,1644,5413,4673,8132,0253,1833,6361,1112...",20.10,25.8
Martin,6,"3,7494,1683,7902,2395,2722,813",19.90,21.8
Washington,4,"4,1032,6532,9511,296",19.55,26.2
Hertford,8,"3,4833,2352,1821,6502,6333,3623,7921,215",19.50,22.3
Jones,3,"3,4312,5323,209",18.80,20.0
Robeson,35,"3,8594,5944,0253,2042,0573,2832,4163,3423,0293...",18.40,23.8
Anson,9,"2,7831,4114,5046492,4759444,4423,4811,366",18.40,21.3


**Plain vs population-weighted mean.** A plain mean treats a tract of 800 people the same as one of 8,000. For a county figure, weight by population. You'll use this again in the build notebook.

In [20]:
wide["diab_people"] = wide["DIABETES"] * wide["TotalPopulation"]   # prevalence x population
weighted = wide.dropna(subset=["DIABETES"]).groupby("CountyName").agg(people=("diab_people", "sum"), pop=("TotalPopulation", "sum"))
weighted["diabetes_weighted"] = weighted["people"] / weighted["pop"]

compare = by_county[["diabetes_median"]].join(weighted[["diabetes_weighted"]])
compare["plain_mean"] = wide.groupby("CountyName")["DIABETES"].mean()
compare.round(1).head(10)

TypeError: can't multiply sequence by non-int of type 'float'

## 5. Merge with `validate`

`validate` is your cardinality check. If you expect each tract to match one county row and it doesn't, pandas raises an error instead of quietly duplicating rows, which is the pandas version of a many-to-many relationship warning.

The county file offers two value types per measure: `CrdPrv` (crude prevalence) and `AgeAdjPrv` (age-adjusted). Tract values are crude, so compare crude with crude.

In [ ]:
county = standardise_columns(pd.read_csv(COUNTY_FILE, dtype=ID_COLS))
print(county["DataValueTypeID"].value_counts())

county_diab = (county.query("MeasureId == 'DIABETES' and DataValueTypeID == 'CrdPrv'")
               [["LocationName", "Data_Value"]]
               .rename(columns={"LocationName": "CountyName", "Data_Value": "county_diabetes"}))

merged = wide.merge(county_diab, on="CountyName", how="left", validate="many_to_one")
print(merged.shape, "| unmatched tracts:", merged["county_diabetes"].isna().sum())

Now see `validate` catch a mistake: forgetting the `CrdPrv` filter leaves two rows per county, so the merge is no longer many-to-one.

In [ ]:
county_both_types = (county.query("MeasureId == 'DIABETES'")[["LocationName", "Data_Value"]]
                     .rename(columns={"LocationName": "CountyName", "Data_Value": "county_diabetes"}))
try:
    wide.merge(county_both_types, on="CountyName", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("Caught:", e)

## 6. Profile

Run these on every new dataset before any analysis. They answer: how big, how complete, what values exist, which years.

In [ ]:
df.info()

In [ ]:
print("Share missing per column (top 10):")
print(df.isna().mean().sort_values(ascending=False).head(10).round(3))
print("\nMeasures:", df["MeasureId"].nunique())
print("\nSurvey years in this release (put this in your README):")
print(df["Year"].value_counts())

In [ ]:
wide[keep + ["TotalPopulation"]].describe().T.round(1)

## Practice: rebuild without looking

In the cell below, starting only from `TRACT_FILE`, rebuild the wide table: correct dtypes, the three measures, a uniqueness check, and one row per tract. Time yourself; under 10 minutes means you're ready for part 2.

The check cell after it tells you whether you got it right.

In [ ]:
# your code here: create a DataFrame called my_wide
my_wide = None

In [ ]:
if my_wide is None:
    print("Not attempted yet.")
else:
    assert set(keep) <= set(my_wide.columns), "missing a measure column"
    assert my_wide["LocationName"].is_unique, "tracts should be unique"
    assert len(my_wide) == len(wide), f"expected {len(wide)} rows, got {len(my_wide)}"
    assert pd.api.types.is_string_dtype(my_wide["LocationName"]), "tract IDs should be strings"
    print("Correct: matches the reference table.")

## What I learned

Write 2–3 sentences before moving on. Anything that confused you goes in your Thursday check-in.

- 